Add Train Model Function

In [7]:
from google.colab import drive
drive.mount('/content/drive')

project_path = "/content/drive/MyDrive/ML Projects/Airfoil/"

train = pd.read_csv(project_path + "train (1).csv")



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [9]:
%%writefile feature_engineering.py
import numpy as np

def feature_engineering(df):
    df = df.copy()

    df["freq_vel_interaction"] = df["frequency"] * df["free-stream-velocity"]
    df["strouhal"] = (df["frequency"] * df["chord-length"]) / df["free-stream-velocity"]
    df["strouhal_squared"] = df["strouhal"] ** 2
    df["strouhal_cubed"] = df["strouhal"] ** 3
    df["strouhal_angle"] = df["strouhal"] * df["attack-angle"]

    df["log_strouhal_angle"] = np.log1p(np.maximum(df["strouhal_angle"], -0.999999))

    df["velocity5_thickness"] = (df["free-stream-velocity"] ** 5) * df["displacement-thickness"]
    df["angle_vel_interaction"] = df["attack-angle"] * df["free-stream-velocity"]
    df["angle_freq_interaction"] = df["attack-angle"] * df["frequency"]
    df["angle_chord_interaction"] = df["attack-angle"] * df["chord-length"]

    df["angle_squared"] = df["attack-angle"] ** 2
    df["angle_freq_squared"] = df["attack-angle"] * (df["frequency"] ** 2)

    df.drop(columns=["velocity5_thickness","strouhal_angle","strouhal"], inplace=True)
    return df


Writing feature_engineering.py


In [10]:
import pandas as pd
import numpy as np

# 📂 Load datasets
train = pd.read_csv('train (1).csv')
test  = pd.read_csv('test (2).csv')

# Columns
X_cols = [
    'frequency','attack-angle','chord-length',
    'free-stream-velocity','displacement-thickness'
]
y_col = 'scaled-sound-pressure'

# Raw X, y
X_train_raw = train[X_cols]
y_train = train[y_col]
X_test_raw  = test[X_cols]

# 🔧 Import your feature engineering function
from feature_engineering import feature_engineering

# Apply FE
X_train = feature_engineering(X_train_raw)
X_test  = feature_engineering(X_test_raw)


In [12]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor

def train_model(X_train, y_train):

    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('xgb', XGBRegressor(objective='reg:squarederror', random_state=1))
    ])

    param_grid = {
        'xgb__n_estimators': [480],
        'xgb__max_depth': [9],
        'xgb__learning_rate': [0.10704067150254538],
        'xgb__subsample': [0.8],
        'xgb__colsample_bytree': [1.0],
        'xgb__reg_alpha': [0.1],
        'xgb__reg_lambda': [10],
        'xgb__min_child_weight': [5],
        'xgb__gamma': [0],
    }

    grid = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        scoring='r2',
        cv=12,
        n_jobs=-1,
        verbose=1
    )

    grid.fit(X_train, y_train)

    train_r2 = grid.best_estimator_.score(X_train, y_train)

    return grid.best_estimator_, grid, train_r2


In [13]:
model, grid_result, train_r2 = train_model(X_train, y_train)

print("Training R²:", train_r2)
print("Best Params:", grid_result.best_params_)


Fitting 12 folds for each of 1 candidates, totalling 12 fits
Training R²: 0.9999773119420516
Best Params: {'xgb__colsample_bytree': 1.0, 'xgb__gamma': 0, 'xgb__learning_rate': 0.10704067150254538, 'xgb__max_depth': 9, 'xgb__min_child_weight': 5, 'xgb__n_estimators': 480, 'xgb__reg_alpha': 0.1, 'xgb__reg_lambda': 10, 'xgb__subsample': 0.8}
